# 03 · Exportación de reglas (todas)

**Proyecto:** Videojuegos y Rendimiento Académico — Ingeniería de Datos, UFRO.

El notebook `02_reglas_asociacion.ipynb` se queda con las reglas de lift ≥ 1.1 (4808) y solo muestra las primeras. Este notebook reproduce la misma minería (mismo dataset discretizado, mismo Apriori) pero **exporta TODAS las reglas, sin filtro de lift**, a un CSV.

- **Entrada:** `data/processed/dataset_discretizado.csv` (generado por el notebook 01).
- **Salida:** `data/processed/reglas_todas.csv` (una fila por regla; columnas `antecedente, consecuente, support, confidence, lift`).

Mismos itemsets frecuentes que `02`: `apriori(min_support=0.05, max_len=4)`. La diferencia es `association_rules(metric='lift', min_threshold=0)`, que no descarta ninguna regla derivable de esos itemsets.

In [1]:
from pathlib import Path

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# raíz del repo: subir hasta el directorio que contiene data/ (funciona desde la raíz o desde cualquier subcarpeta de notebooks/)
REPO = Path.cwd().resolve()
while not (REPO / 'data').is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
PROCESSED = REPO / 'data' / 'processed' / 'dataset_discretizado.csv'
SALIDA = REPO / 'data' / 'processed' / 'reglas_todas.csv'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df = pd.read_csv(PROCESSED)
print('Dimensiones:', df.shape)
df.head()

Dimensiones: (7865, 13)


,gender,gaming_genre,stress_level,age,gaming_hours,study_hours,sleep_hours,attendance,social_activity,device_usage,reaction_time_ms,addiction_score,grades
0,Male,FPS,Low,Q3,Q4,Q4,Q3,Q4,Q3,Q3,Q1,Q4,Q4
1,Male,Casual,Medium,Q2,Q1,Q4,Q3,Q1,Q1,Q1,Q4,Q1,Q4
2,Female,Casual,High,Q4,Q1,Q4,Q1,Q3,Q3,Q1,Q4,Q1,Q4
3,Female,RPG,Low,Q2,Q4,Q1,Q4,Q2,Q2,Q4,Q1,Q4,Q1
4,Female,FPS,Low,Q3,Q3,Q3,Q2,Q1,Q1,Q3,Q2,Q3,Q2


## Minería de reglas (sin filtro de lift)

One-hot → itemsets frecuentes (Apriori) → todas las reglas (`min_threshold=0`). Las aserciones fijan los conteos esperados: 7886 reglas en total, de las cuales 4808 mantienen lift ≥ 1.1 (las que reporta el notebook 02). Si el dataset o los parámetros cambian, el notebook falla en vez de exportar algo distinto en silencio.

In [2]:
onehot = pd.get_dummies(df, prefix_sep='=').astype(bool)
frecuentes = apriori(onehot, min_support=0.05, use_colnames=True, max_len=4)
reglas = association_rules(frecuentes, metric='lift', min_threshold=0)

print('Itemsets frecuentes:', len(frecuentes))
print('Reglas (todas):', len(reglas))
print('Reglas con lift >= 1.1:', int((reglas['lift'] >= 1.1).sum()))
assert len(reglas) == 7886, f'Se esperaban 7886 reglas, se obtuvieron {len(reglas)}'
assert int((reglas['lift'] >= 1.1).sum()) == 4808, 'El subconjunto lift >= 1.1 ya no es 4808'

Itemsets frecuentes: 1767
Reglas (todas): 7886
Reglas con lift >= 1.1: 4808


## Formato y exportación

Los `frozenset` de antecedente/consecuente se aplanan a texto legible (`item + item`) y se conservan las tres métricas. Se ordena por lift descendente.

In [3]:
def fmt(itemset):
    return ' + '.join(sorted(itemset))

reglas['antecedente'] = reglas['antecedents'].apply(fmt)
reglas['consecuente'] = reglas['consequents'].apply(fmt)
cols = ['antecedente', 'consecuente', 'support', 'confidence', 'lift']

export = reglas.sort_values('lift', ascending=False)[cols].reset_index(drop=True)
export.to_csv(SALIDA, index=False)
print('Guardado:', SALIDA, '|', len(export), 'reglas')
export.head(15).round(3)

Guardado: /home/strawberry/Projects/Gaming-vs-Academic/data/processed/reglas_todas.csv | 7886 reglas


,antecedente,consecuente,support,confidence,lift
0,stress_level=High,sleep_hours=Q1 + study_hours=Q4,0.062,0.494,7.961
1,sleep_hours=Q1 + study_hours=Q4,stress_level=High,0.062,1.000,7.961
2,grades=Q4 + reaction_time_ms=Q4,gaming_hours=Q1 + study_hours=Q4,0.051,0.441,7.158
3,gaming_hours=Q1 + study_hours=Q4,grades=Q4 + reaction_time_ms=Q4,0.051,0.823,7.158
4,gaming_hours=Q1 + grades=Q4,reaction_time_ms=Q4 + study_hours=Q4,0.051,0.418,6.755
5,reaction_time_ms=Q4 + study_hours=Q4,gaming_hours=Q1 + grades=Q4,0.051,0.819,6.755
6,grades=Q1 + reaction_time_ms=Q1,gaming_hours=Q4 + study_hours=Q1,0.050,0.385,6.328
7,gaming_hours=Q4 + study_hours=Q1,grades=Q1 + reaction_time_ms=Q1,0.050,0.826,6.328
8,reaction_time_ms=Q1 + study_hours=Q1,gaming_hours=Q4 + grades=Q1,0.050,0.825,6.016
9,gaming_hours=Q4 + grades=Q1,reaction_time_ms=Q1 + study_hours=Q1,0.050,0.366,6.016
